# BYTE - kendi sesinle toplu seslendirme (Chatterbox Turbo)

Bu defter **senin sesinin** kayit ornegiyle, siradaki senaryolarin HER cumlesini okutur ve sesleri **ozel R2 alanina** koyar.
`daily` render'dan once o senaryonun sesini bulursa Piper yerine senin sesini kullanir; bulamazsa eski sesle devam eder.

**Calisma zamani:** Calisma zamani > Calisma zamani turunu degistir > **T4 GPU**. A100 gerekmez.

**Sira:** 1 kurulum - 2 ayarlar - 3 ses ornegini yukle - 4 modeli yukle ve 3 cumleyi dinle - 5 toplu uret - 6 dinle (istege bagli) - 7 R2'ye gonder - 8 temizle.
Ilk seferde `HOW_MANY = 2` yap, sesi dinle, begenirsen 10-20'ye cikar.

**Gizlilik**
- Ses ornegin yalniz bu oturumda kalir; 8. hucre siler. **Hicbir yere (GitHub dahil) yukleme**: baskasi sesini klonlayabilir.
- R2 anahtarlarini hucrelere YAZMA. Sol menude anahtar simgesi (Secrets) > su 4 isimle ekle ve "Notebook access" acik olsun:
  `R2_ENDPOINT`, `R2_BUCKET`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`.
  (Secret key yalniz olustururken gorunur; kaybettiysen Cloudflare > R2 > Manage API tokens > bucket'a "Object Read & Write" yeni token.)

**Kayit tarifi (10-20 sn, telefon yeter)**
- Sessiz oda, mikrofon agzindan ~20 cm, yankili (bos, buyuk) oda degil, arkada muzik/TV yok.
- Dogal anlatici gibi oku: video anlatiyormus gibi, sarki gibi okuma. Cumle ortasinda dur-kalk yapma, kekeleme olursa bastan al.
- Ingilizce oku (kanal Ingilizce; aksan sesine gecer). Su metni oku:
  *"One second of video costs more than this whole channel. A model reads tokens, not words, and every token has a price. Here is the number, measured, and here is where it comes from."*

In [ ]:
#@title 1) Kurulum (birkac dakika)
!pip -q install chatterbox-tts boto3
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/SemihKopcal/byte-shorts /content/repo
# Colab "oturumu yeniden baslat" derse: Calisma zamani > Oturumu yeniden baslat, sonra 2. hucreden devam (repo klasoru kalir).
print("tamam")

In [ ]:
#@title 2) Ayarlar
HOW_MANY = 2            #@param {type:"integer"}
SCRIPT_IDS = ""         #@param {type:"string"}
SPEAKER = "semih"       #@param {type:"string"}
SEED = 7                #@param {type:"integer"}
# HOW_MANY: kuyrukta SIRADA bekleyen ilk N senaryo. SCRIPT_IDS doluysa (virgullu id'ler) o gecerli.

In [ ]:
#@title 3) Ses ornegini yukle (10-20 sn, wav/mp3/m4a)
import subprocess
from google.colab import files
up = files.upload()
ref_raw = next(iter(up))
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", ref_raw, "-ac", "1", "-ar", "24000", "-t", "25",
                "-af", "highpass=f=70,loudnorm=I=-20:TP=-2:LRA=11", "/content/ref.wav"], check=True)
dur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                                     "-of", "default=nw=1:nk=1", "/content/ref.wav"], text=True))
print(f"hazir: {ref_raw} -> /content/ref.wav ({dur:.1f} sn)")
if dur < 8:
    print("UYARI: 8 sn'den kisa; 10-20 sn kayit daha iyi sonuc verir.")

In [ ]:
#@title 4) Modeli yukle ve 3 cumleyi dinle
import time, torch, torchaudio as ta
from IPython.display import Audio, display
from chatterbox.tts_turbo import ChatterboxTurboTTS

try:                                    # HF anahtari GEREKMEZSE bu hucre sessizce gecer
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
except Exception:
    pass

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "GPU yok: Calisma zamani > Calisma zamani turunu degistir > T4 GPU"
turbo = ChatterboxTurboTTS.from_pretrained(device=dev)
REF = "/content/ref.wav"
for k, txt in {"a": "This is BYTE. Every number on this screen was measured, not guessed.",
               "b": "A token is about four characters, and every one of them has a price.",
               "c": "That comes to about seven hundred fifty megabytes."}.items():
    t = time.time()
    torch.manual_seed(SEED)
    wav = turbo.generate(txt, audio_prompt_path=REF)
    ta.save(f"/content/deneme_{k}.wav", wav.cpu(), turbo.sr)
    print(f"{k}: {time.time()-t:.1f} sn  | {txt}")
    display(Audio(f"/content/deneme_{k}.wav"))
# Kendi sesine benzemiyorsa ya da cok yabanci aksanliysa dur: daha temiz/uzun bir kayitla 3. hucreyi tekrarla.

In [ ]:
#@title 5) Toplu uret (HOW_MANY senaryo)
import hashlib, json, shutil, subprocess, time
from pathlib import Path

REPO = "/content/repo"
OUT = Path("/content/voice_out")
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)

def sh(*args):
    return subprocess.check_output(["python", *args], cwd=REPO, text=True)

ids = [s.strip() for s in SCRIPT_IDS.split(",") if s.strip()] or sh("tools/voice_lines.py", "--next", str(HOW_MANY)).split()
print("uretilecek:", ids)

def words(s): return max(1, len(s.split()))
def plausible(sec, say):
    per = sec / words(say)                  # cumle basina saniye/kelime: ~0.25-0.6 normal; cok uzun = takilma/hayal
    return 0.12 <= per <= 0.75

cache = {}                                  # ayni cumle (CTA) bir kez uretilir
flags = []
t_all = time.time()
for sid in ids:
    info = json.loads(sh("tools/voice_lines.py", sid))
    d = OUT / info["id"]
    d.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for b in info["beats"]:
        say, path = b["say"], d / f"{b['id']}.wav"
        if say in cache:
            shutil.copy(cache[say], path)
            continue
        base = int(hashlib.sha1(say.encode()).hexdigest()[:6], 16)
        for attempt in range(4):            # tutarsiz uzunluk cikarsa baska tohumla tekrar dene
            torch.manual_seed(SEED + base % 10000 + attempt * 101)
            wav = turbo.generate(say, audio_prompt_path=REF).cpu()
            sec = wav.shape[-1] / turbo.sr
            if plausible(sec, say):
                break
        else:
            flags.append((info["id"], b["id"], round(sec, 1), say))
        ta.save(str(path), wav, turbo.sr)
        cache[say] = path
    (d / "manifest.json").write_text(json.dumps({**info["manifest"], "voice": SPEAKER}))
    print(f"  {info['id']}: {len(info['beats'])} cumle, {time.time()-t0:.0f} sn")
print(f"bitti: {len(ids)} senaryo, {time.time()-t_all:.0f} sn")
if flags:
    print("DIKKAT: su cumleler sureleri tuhaf cikti (dinle):")
    for f in flags:
        print("  ", f)

In [ ]:
#@title 6) Dinle (istege bagli): bir senaryonun cumlelerini sirayla
LISTEN = ""   #@param {type:"string"}
sid = LISTEN or sorted(p.name for p in OUT.iterdir())[0]
man = json.loads((OUT / sid / "manifest.json").read_text())
info = json.loads(sh("tools/voice_lines.py", sid))
print(sid)
for b in info["beats"]:
    print(f"{b['id']}: {b['say']}")
    display(Audio(str(OUT / sid / f"{b['id']}.wav")))

In [ ]:
#@title 7) Ozel R2 alanina gonder (Secrets gerekir)
import os
from google.colab import userdata
need = ["R2_ENDPOINT", "R2_BUCKET", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"]
missing = []
for k in need:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        missing.append(k)
if missing:
    print("Secrets eksik:", ", ".join(missing))
    print("Sol menu > anahtar simgesi > bu isimlerle ekle, 'Notebook access' ac, sonra bu hucreyi tekrar calistir.")
else:
    print(subprocess.run(["python", "tools/r2_voice.py", "push", str(OUT)], cwd=REPO, capture_output=True, text=True,
                         env=os.environ).stdout or "bos cikti")
    print(subprocess.run(["python", "tools/r2_voice.py", "list"], cwd=REPO, capture_output=True, text=True,
                         env=os.environ).stdout)

In [ ]:
#@title 8) Temizle: ses ornegini sil ve calisma zamanini kapat
import os, shutil
from google.colab import runtime
for p in ("/content/ref.wav", "/content/deneme_a.wav", "/content/deneme_b.wav", "/content/deneme_c.wav"):
    if os.path.exists(p):
        os.remove(p)
for f in list(up):                      # yuklenen ham kayit
    if os.path.exists(f):
        os.remove(f)
shutil.rmtree("/content/voice_out", ignore_errors=True)
print("silindi. R2'de gorunmeyen sey olduysa 7. hucreyi tekrar calistirmadan kapatma!")
runtime.unassign()                      # acik kalan GPU birim yakar